# Capsule Wardrobe Planner

**Model:** `AlexWortega/openjev` (`qwen3.5-2b-nli-v5`)

**Task:** to plan outfits for a week from a small wardrobe (3 tops, 3 bottoms, 2 outerwear items), given each day's weather and dress code. Outerwear is added to the candidates only on cold, rainy, snowy or windy days.

**How an outfit is scored:**
- Premise: a description of the outfit, including the fabric, warmth and formality of each item.
- Two claims are checked against it: the outfit suits the weather, and the outfit fits the dress code.
- Score = average of the two entailment probabilities minus a re-wear penalty (0.05 for every earlier wear of each item).

## System 1 and System 2 definition

**System 1 (fast, intuitive):** the NLI model. One forward pass returns an entailment probability for a claim. It has no memory and judges each outfit on its own.

**System 2 (slow, deliberate):** the planning loop around the model. It keeps state from day to day (recent combinations, items worn so far), removes recent repeats, penalizes re-wearing, rejects outfits with a claim score below `0.3` (warning is given if none pass) and picks the best remaining outfit. It works as a greedy day-by-day search.

In [25]:
import os
from collections import Counter
from dataclasses import dataclass
from typing import List
from itertools import product

import torch
from huggingface_hub import snapshot_download
from transformers import AutoModelForSequenceClassification, AutoTokenizer

In [26]:
model_repo = "AlexWortega/openjev"
subfolder = "qwen3.5-2b-nli-v5"

In [27]:
class JevScorer:
  def __init__(self, subfolder=subfolder, batch_size=32):
      self.device = "cuda" if torch.cuda.is_available() else "cpu"
      self.batch_size = batch_size
      self.cache = {}

      root = snapshot_download(model_repo, allow_patterns=[f"{subfolder}/*"])
      path = os.path.join(root, subfolder)
      dtype = torch.float32
      if self.device == "cuda":
          dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

      self.tok = AutoTokenizer.from_pretrained(path)
      if self.tok.pad_token is None:
          self.tok.pad_token = self.tok.eos_token
      self.model = AutoModelForSequenceClassification.from_pretrained(
          path, dtype=dtype, device_map={"": self.device}
      ).eval()
      self.model.config.pad_token_id = self.tok.pad_token_id

      id2label = {int(k): str(v).lower() for k, v in self.model.config.id2label.items()}
      self.ent_idx = next((k for k, v in id2label.items() if v.startswith("entail")), 1)

  @torch.inference_mode()
  def score(self, pairs):
      todo = sorted({p for p in pairs if p not in self.cache}, key=lambda p: len(p[0]) + len(p[1]))
      for i in range(0, len(todo), self.batch_size):
          chunk = todo[i:i + self.batch_size]
          texts = [self.model.config.nli_template.format(premise=p, hypothesis=h) for p, h in chunk]
          enc = self.tok(texts, return_tensors="pt", padding=True, truncation=True).to(self.device)
          probs = torch.softmax(self.model(**enc).logits.float(), dim=-1)[:, self.ent_idx].tolist()
          self.cache.update(zip(chunk, probs))
      return [self.cache[p] for p in pairs]

In [29]:
@dataclass(frozen=True)
class ClothesItem:
  name: str
  description: str   # "a crisp white button-down shirt"
  traits: str = ""   # fabric, warmth, formality, water resistance

@dataclass
class DayResult:
  day: str
  outfit: tuple
  weather_p_ent: float
  dress_p_ent: float
  score: float           # average of the two, minus the re-wear penalty

OUTER_WEATHER = {"cold", "light rain", "heavy rain", "snowy", "windy", "cold and rainy"}


def describe(outfit):
  """
  Creates an explicit, information-rich premise for the NLI model
  """
  parts = []
  for item in outfit:
    if item is None:
      continue
    text = (f"The outfit includes {item.description}. The clothing is {item.traits}.")
    parts.append(text)
  return " ".join(parts)


def candidates(weather, tops, bottoms, outers):
  outer_options = outers if weather in OUTER_WEATHER else [None]
  return list(product(tops, bottoms, outer_options))

In [30]:
def _item_names(outfit):
  """
  Names the real items in an outfit (drops the None outer slot)
  """
  return [item.name for item in outfit if item is not None]


def _combo(outfit):
  """
  (top name, bottom name): what the no-repeat rule compares
  """
  top, bottom, _outer = outfit
  return (top.name, bottom.name)


def _claims(weather, dress_code):
  """
  Builds the weather claim and the dress-code claim
  """
  weather_claim = (
      f"This outfit provides appropriate clothing protection "
      f"for {weather} weather."
  )
  dress_claim = f"This outfit is appropriate for a {dress_code} dress code."
  return weather_claim, dress_claim


def _pick_options(outfits, recent_combos):
  """
  Drops recently worn combos, unless that would leave nothing
  """
  fresh = [o for o in outfits if _combo(o) not in recent_combos]
  return fresh or outfits


def _build_pairs(options, weather_claim, dress_claim):
  """
  Creates two (premise, hypothesis) pairs per outfit: weather and dress
  """
  pairs = []

  for outfit in options:
      premise = describe(outfit)
      pairs.append((premise, weather_claim))
      pairs.append((premise, dress_claim))

  return pairs


def _rewear_cost(outfit, worn, rewear_penalty):
  """
  Assigns penalty that grows with how often the outfit's items were already worn
  """
  times_worn = sum(worn[name] for name in _item_names(outfit))
  return rewear_penalty * times_worn

In [32]:
def plan_day(scorer, day, weather, dress_code, tops, bottoms, outers,
             recent_combos=(), worn=None,
             rewear_penalty=0.05, min_claim_score=0.3) -> DayResult:
  """
  Picks the best outfit for one day
  recent_combos:
    Clothing combinations that should preferably not be repeated
  worn:
    Counter with the number of times each clothing item was worn
  min_claim_score:
    Minimum acceptable score for both weather and dress
  """
  worn = worn or Counter()
  weather_claim, dress_claim = _claims(weather, dress_code)

  outfits = candidates(weather, tops, bottoms, outers)
  options = _pick_options(outfits, recent_combos)

  pairs = _build_pairs(options, weather_claim, dress_claim)
  scores = scorer.score(pairs)
  weather_scores = scores[0::2]
  dress_scores = scores[1::2]

  scored = []

  for i, outfit in enumerate(options):
      weather_score = weather_scores[i]
      dress_score = dress_scores[i]
      cost = _rewear_cost(outfit, worn, rewear_penalty)

      scored.append(
          DayResult(
              day=day,
              outfit=outfit,
              weather_p_ent=weather_score,
              dress_p_ent=dress_score,
              score=(weather_score + dress_score) / 2 - cost,
          )
      )

  passing = [
      r for r in scored
      if r.weather_p_ent >= min_claim_score
      and r.dress_p_ent >= min_claim_score
  ]

  if not passing:
      print(f"Warning: no outfit reached the minimum entailment score for {weather} weather and {dress_code} dress code on {day}.")
      passing = scored

  return max(passing, key=lambda r: r.score)


def plan_week(scorer, week, tops, bottoms, outers, no_repeat_days=2,
              rewear_penalty=0.05, min_claim_score=0.3) -> List[DayResult]:
  """
  Plans a whole week, one day at a time, remembering what was worn
  week:
    List of (day, weather, dress_code) tuples
  """
  history = []
  worn = Counter()
  plan = []

  for day, weather, dress_code in week:
      recent = history[-no_repeat_days:] if no_repeat_days > 0 else []

      result = plan_day(scorer, day, weather, dress_code, tops, bottoms,
                        outers, recent_combos=recent, worn=worn,
                        rewear_penalty=rewear_penalty,
                        min_claim_score=min_claim_score)

      history.append(_combo(result.outfit))
      worn.update(_item_names(result.outfit))
      plan.append(result)
  return plan

In [33]:
scorer = JevScorer()

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/618 [00:00<?, ?it/s]

### Demo

In [34]:
tops = [
  ClothesItem(
      "white_shirt",
      "a crisp white button-down shirt",
      "lightweight cotton and formal; provides little warmth and no meaningful rain protection",
  ),

  ClothesItem(
      "grey_tee",
      "a plain grey crewneck t-shirt",
      "thin cotton and very casual; provides little warmth and no rain protection",
  ),

  ClothesItem(
      "black_sweater",
      "a fitted black knit sweater",
      "warm wool-blend knit and smart; provides substantial warmth but little rain protection",
  ),
]


bottoms = [
  ClothesItem(
      "navy_trousers",
      "tailored navy trousers",
      "mid-weight wool-blend and formal; provides moderate warmth and no rain protection",
  ),

  ClothesItem(
      "dark_jeans",
      "dark wash straight-leg jeans",
      "thick denim and casual; provides moderate warmth but no meaningful rain protection",
  ),

  ClothesItem(
      "black_skirt",
      "a knee-length black pencil skirt",
      "thin fabric and formal; provides little warmth and no rain protection",
  ),
]


outers = [
  ClothesItem(
      "trench_coat",
      "a beige trench coat",
      "waterproof and wind-proof; provides strong protection from rain and wind and is suitable for wet weather",
  ),

  ClothesItem(
      "puffer",
      "a black puffer jacket",
      "heavily insulated and very warm, water-resistant and wind-protective; provides strong warmth and some protection from rain",
  ),
]


week = [
  ("Monday", "mild", "business casual"),
  ("Tuesday", "cold and rainy", "business casual"),
  ("Wednesday", "sunny", "smart"),
  ("Thursday", "heavy rain", "business formal"),
  ("Friday", "mild", "casual"),
]

In [39]:
def report_result(r: DayResult) -> None:
    """
    Prints one day's chosen outfit and its scores
    """
    top, bottom, outer = r.outfit
    outer_name = outer.name if outer else "no outerwear"

    print(
        f"{r.day}: "
        f"{top.name} + {bottom.name} + {outer_name}  "
        f"weather_score={r.weather_p_ent:.2f} "
        f"dress_score={r.dress_p_ent:.2f} "
        f"total={r.score:.2f}"
    )

A whole week plan

In [41]:
results = plan_week(scorer, week, tops, bottoms, outers)
for r in results:
  report_result(r)

Monday: black_sweater + navy_trousers + no outerwear  weather_score=0.96 dress_score=0.53 total=0.74
Tuesday: white_shirt + black_skirt + trench_coat  weather_score=0.68 dress_score=0.33 total=0.50
Wednesday: black_sweater + dark_jeans + no outerwear  weather_score=0.91 dress_score=0.64 total=0.73
Thursday: white_shirt + navy_trousers + trench_coat  weather_score=0.97 dress_score=0.68 total=0.68
Friday: grey_tee + dark_jeans + no outerwear  weather_score=0.61 dress_score=0.90 total=0.70


One day plan with no previous history

In [43]:
r_1 = plan_day(scorer, "Saturday", "light rain", "business", tops, bottoms, outers)
report_result(r_1)

Saturday: white_shirt + navy_trousers + trench_coat  weather_score=0.67 dress_score=0.53 total=0.60


One day plan & a note about previous day

In [44]:
r_2 = plan_day(scorer, "Saturday", "light rain", "business", tops, bottoms, outers,
             recent_combos=[("white_shirt", "navy_trousers")], worn=Counter({"navy_trousers": 2}))
report_result(r_2)

Saturday: white_shirt + black_skirt + trench_coat  weather_score=0.52 dress_score=0.35 total=0.44


## Conclusion

**What the output shows**
* Weather scores are mostly high (0.61 to 0.97). Thursday (heavy rain) reached 0.97 with the "waterproof" `trench_coat`.
* Dress-code scores are lower and vary more. Friday ("casual") scored highest at 0.90 with every other day staying between 0.33 and 0.68.
* Tuesday (cold and rainy) is the weakest day: total 0.50, dress score 0.33.
* The planner's memory works. Thursday's total was reduced by about 0.15 because its three items had been worn earlier. In the Saturday test, blocking `navy_trousers` lowered the total from 0.60 to 0.44.

**Likely reasons**
* The weather claim ("appropriate clothing protection") matches wording such as "waterproof" or "warm" in the item descriptions, so protective items score high. A warm sweater scored 0.91 for "sunny" weather, while a thin t-shirt scored only 0.61 for "mild". The claim seems to reward protection rather than suitability.
* The item descriptions only say "formal", "casual" or "smart", so mixed codes such as "business casual" have no direct match. Moreover, "casual" matches exactly, and it fully explains the 0.90.
* On Tuesday no outfit covers both cold and rain: the trench coat is described for rain and wind but not warmth, and the puffer is warm but only partly rain-protective. The model still gave a thin shirt and skirt under the trench coat 0.68 for weather, so it appears to give most of the weight to the coat.
* Avoiding repeats has a cost. With only 8 items, blocking the best combination forces a weaker outfit, as in the Saturday test.

**Limits:** a small 2B model, five days of history, immediate penalty imposing when the item is repeated.